# Image Captioning with Visual Attention — Colab Runner

This notebook runs the full pipeline (data prep -> feature extraction -> training -> evaluation -> demo) using the project's `.py` modules.

**Before running:** Runtime -> Change runtime type -> GPU.

**Setup you need to do once:**
1. Download the Flickr8k dataset (images + text) — e.g. from Kaggle (`kaggle datasets download -d adityajn105/flickr8k`) or the original source.
2. Put it in your Google Drive as:
   - `MyDrive/image-captioning/data/Flickr8k_text/` (contains `Flickr8k.token.txt`, `Flickr_8k.trainImages.txt`, `Flickr_8k.testImages.txt`)
   - `MyDrive/image-captioning/data/Flicker8k_Dataset/` (the .jpg images)
3. Upload this project's `.py` files into `MyDrive/image-captioning/` as well (or clone your GitHub repo once you've pushed it there).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/image-captioning'
%cd $PROJECT_ROOT

In [ ]:
!pip install -q -r requirements.txt
import nltk
nltk.download('punkt')

In [ ]:
import os
os.environ['CAPTION_PROJECT_ROOT'] = PROJECT_ROOT

import importlib, config
importlib.reload(config)
print('Text dir:', config.DATASET_TEXT_DIR)
print('Images dir:', config.DATASET_IMAGES_DIR)
print('Artifacts dir:', config.ARTIFACTS_DIR)

## Step 1 — Prepare captions + tokenizer

In [ ]:
import data_prep
train_descriptions, test_descriptions, tokenizer, max_len = data_prep.prepare_all()

## Step 2 — Extract & cache CNN features (one-time, GPU accelerated)

In [ ]:
import features
all_images = list(train_descriptions.keys()) + list(test_descriptions.keys())
features.extract_and_cache_features(all_images, config.DATASET_IMAGES_DIR, config.FEATURES_DIR)

## Step 3 — Train the attention model

In [ ]:
import train
train.main()

## Step 4 — Evaluate with BLEU on the test set

In [ ]:
import evaluate
results = evaluate.evaluate(method='beam', sample_size=300)  # use sample_size=None for the full test set

## Step 5 — Try it on a single image + view attention

In [ ]:
from predict import CaptionPredictor, plot_attention
import os

predictor = CaptionPredictor()
sample_img = os.path.join(config.DATASET_IMAGES_DIR, list(test_descriptions.keys())[0])

caption, attn = predictor.generate_greedy(sample_img)
print('Caption:', caption)
plot_attention(sample_img, caption.split(), attn)

## Step 6 — Launch the Gradio demo (shareable link)

In [ ]:
import app
app.demo.launch(share=True)